# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

In [ ]:
# imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# using ollama and open ai's models
llama_model = "llama3.2"
gpt_model = "gpt-5-nano"
o4_model = "o4-mini-2025-04-16"

ollama_url = "http://localhost:11434/"
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
requests.get("http://localhost:11434/").content


load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [ ]:

alex_prompt = """
You are Alex, a narcissist who thinks charlie is a narcissist but is unaware of his own flaw.
You are in a conversation with Blake and Charlie.
"""

blake_prompt = """
You are Blake, a therapist.
You are in a conversation with Alex and Charlie.
"""

charlie_prompt = """
You are Charlie, a narcissist who thinks alex is a narcissist but is unaware of his own flaw.
You are in a conversation with Blake and Alex.
"""

# starting convo
alex_messages = ["Hi there"]
blake_messages = ["Hi"]
charlie_messages = ["What?"]

In [ ]:
# each model sees itself as assistant and others as user
# this is so it knows it has to reply something.

def call_llama():
    # each model follows its own system prompt, in this case, 'alex_prompt'
    messages = [{"role": "system", "content": alex_prompt}]
    
    for alex, blake, charlie in zip(alex_messages, blake_messages, charlie_messages):
        messages.append({"role": "assistant", "content": alex})
        messages.append({"role": "user", "content": blake})
        messages.append({"role": "user", "content": charlie})
    response = ollama.chat.completions.create(model=llama_model, messages=messages)
    return response.choices[0].message.content

def call_gpt():
    messages = [{"role": "system", "content": blake_prompt}]
    for alex, blake, charlie in zip(alex_messages, blake_messages, charlie_messages):
        messages.append({"role": "user", "content": alex})
        messages.append({"role": "assistant", "content": blake})
        messages.append({"role": "user", "content": charlie})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

def call_o4():
    messages = [{"role": "system", "content": charlie_prompt}]
    for alex, blake, charlie in zip(alex_messages, blake_messages, charlie_messages):
        messages.append({"role": "user", "content": alex})
        messages.append({"role": "user", "content": blake})
        messages.append({"role": "assistant", "content": charlie})
    response = openai.chat.completions.create(model=o4_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
display(Markdown(f"### Alex:\n{alex_messages[0]}\n"))
display(Markdown(f"### Blake:\n{blake_messages[0]}\n"))
display(Markdown(f"### Charlie:\n{charlie_messages[0]}\n"))

for i in range(5):
    alex_next = call_llama()
    display(Markdown(f"### Alex:\n{alex_next}\n"))
    alex_messages.append(alex_next)
    
    blake_next = call_gpt()
    display(Markdown(f"### Blake:\n{blake_next}\n"))
    blake_messages.append(blake_next)

    charlie_next = call_o4()
    display(Markdown(f"### Charlie:\n{charlie_next}\n"))
    charlie_messages.append(charlie_next)